# 🌐 LG Gram Reddit Pros 분석 한글 번역

## 프로젝트 개요
LG Gram Reddit 분석에서 추출된 Pros(장점) 분석 데이터를 Google Translate API를 활용하여 한글로 번역하는 노트북입니다.

### 주요 기능
1. **Pros 데이터 로드**: `pros_analysis.csv` 파일 읽기
2. **Deep Translator API**: `deep-translator` 라이브러리 활용
3. **배치 번역**: 대량 데이터 효율적 처리
4. **한글 파일 생성**: 번역 결과를 새 CSV 파일로 저장
5. **품질 검증**: 번역 품질 확인 및 통계

### 번역 전략
- **배치 처리**: API 제한을 고려한 효율적 번역
- **오류 처리**: 번역 실패 시 원문 유지
- **진행률 표시**: 실시간 번역 진행상황 확인

In [1]:
# 1. 필수 라이브러리 임포트 및 설정

# 기본 라이브러리
import pandas as pd
import numpy as np
from pathlib import Path
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 번역 라이브러리
try:
    from deep_translator import GoogleTranslator
    print("✅ Deep Translator 라이브러리 사용 가능")
except ImportError:
    print("❌ Deep Translator 설치 필요")
    print("터미널에서 다음 명령어를 실행하세요: pip install deep-translator")

# 시각화 라이브러리
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # 한글 폰트 설정
    plt.rcParams['font.family'] = ['Malgun Gothic', 'AppleGothic', 'Noto Sans CJK KR', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    print("✅ 시각화 라이브러리 및 한글 폰트 설정 완료")
except ImportError:
    print("❌ matplotlib, seaborn 설치 필요")

print("🚀 라이브러리 임포트 완료!")
print(f"📅 실행 시간: {time.strftime('%Y-%m-%d %H:%M:%S')}")

✅ Deep Translator 라이브러리 사용 가능
✅ 시각화 라이브러리 및 한글 폰트 설정 완료
🚀 라이브러리 임포트 완료!
📅 실행 시간: 2025-08-25 15:50:38
✅ 시각화 라이브러리 및 한글 폰트 설정 완료
🚀 라이브러리 임포트 완료!
📅 실행 시간: 2025-08-25 15:50:38


In [2]:
# 2. 파일 경로 설정 및 Pros 데이터 로드

import os

# 현재 노트북 위치 기준으로 절대 경로 설정
current_dir = Path.cwd()
print(f"📍 현재 디렉토리: {current_dir}")

# 데이터 파일 경로 설정 (절대 경로 사용)
data_dir = Path("c:/Users/lgdx/LG_DX_School/03.CX_Group4/02.LG_Gram/data/lg_gram_reddit/analysis")
pros_file_path = data_dir / 'pros_analysis.csv'
korean_pros_file = data_dir / 'Pros_LG_Korean.csv'

print(f"📁 데이터 디렉토리: {data_dir}")
print(f"📂 입력 파일: {pros_file_path}")
print(f"📄 출력 파일: {korean_pros_file}")

# 파일 존재 확인
print(f"🔍 파일 존재 확인...")
print(f"   - 파일 존재: {pros_file_path.exists()}")

if pros_file_path.exists():
    print("✅ pros_analysis.csv 파일 발견")
    
    try:
        # 데이터 로드
        print("📖 Pros 분석 데이터 로드 중...")
        df_pros = pd.read_csv(pros_file_path, encoding='utf-8-sig')
        
        print(f"📊 데이터 정보:")
        print(f"   - 총 행 수: {len(df_pros):,}")
        print(f"   - 컬럼: {list(df_pros.columns)}")
        
        # 데이터 샘플 확인 (처음 3개만)
        print(f"\n📝 데이터 샘플:")
        sample_data = df_pros.head(3)
        for i, (idx, row) in enumerate(sample_data.iterrows()):
            sentence = str(row['pros_sentence'])[:80]  # 80자로 제한
            print(f"   {i+1}. {sentence}...")
        
        # 기본 통계만 출력
        print(f"\n📈 데이터 기본 통계:")
        sentence_lengths = df_pros['pros_sentence'].str.len()
        print(f"   - 평균 문장 길이: {sentence_lengths.mean():.1f}자")
        print(f"   - 최대 문장 길이: {sentence_lengths.max()}자")
        print(f"   - 최소 문장 길이: {sentence_lengths.min()}자")
        print(f"   - NULL 값: {df_pros['pros_sentence'].isnull().sum()}개")
        
        print("✅ 데이터 로드 완료!")
        
    except Exception as e:
        print(f"❌ 데이터 로드 중 오류: {e}")
        df_pros = None
        
else:
    print("❌ pros_analysis.csv 파일을 찾을 수 없습니다.")
    print(f"   확인된 경로: {pros_file_path}")
    # 대체 경로들 확인
    alt_paths = [
        Path("c:/Users/lgdx/LG_DX_School/03.CX_Group4/02.LG_Gram/pros_analysis.csv"),
        Path("c:/Users/lgdx/LG_DX_School/pros_analysis.csv"),
        current_dir / "pros_analysis.csv"
    ]
    
    print("🔍 대체 경로 확인 중...")
    for alt_path in alt_paths:
        if alt_path.exists():
            print(f"   ✅ 발견: {alt_path}")
            pros_file_path = alt_path
            break
        else:
            print(f"   ❌ 없음: {alt_path}")
    
    df_pros = None

📍 현재 디렉토리: c:\Users\lgdx\LG_DX_School\03.CX_Group4\02.LG_Gram\reddit_crawler
📁 데이터 디렉토리: c:\Users\lgdx\LG_DX_School\03.CX_Group4\02.LG_Gram\data\lg_gram_reddit\analysis
📂 입력 파일: c:\Users\lgdx\LG_DX_School\03.CX_Group4\02.LG_Gram\data\lg_gram_reddit\analysis\pros_analysis.csv
📄 출력 파일: c:\Users\lgdx\LG_DX_School\03.CX_Group4\02.LG_Gram\data\lg_gram_reddit\analysis\Pros_LG_Korean.csv
🔍 파일 존재 확인...
   - 파일 존재: True
✅ pros_analysis.csv 파일 발견
📖 Pros 분석 데이터 로드 중...
📊 데이터 정보:
   - 총 행 수: 1,129
   - 컬럼: ['pros_sentence']

📝 데이터 샘플:
   1. My options seem to be a LG Gram 16, but I do not remember anymore what model was...
   2. Tuning LG Gram Pro 17 for productivity? Lack Snappy feeling..  Any Tips?...
   3. I have LG Gram Pro 17 32GB Ultra 7. Even setting FAN to HIGH, Turned off Energy ...

📈 데이터 기본 통계:
   - 평균 문장 길이: 238.9자
   - 최대 문장 길이: 1695자
   - 최소 문장 길이: 4자
   - NULL 값: 0개
✅ 데이터 로드 완료!


In [3]:
# 3. Google Translate 번역 함수 정의 (deep-translator 사용)

def translate_text_to_korean(text, translator):
    """
    개별 텍스트를 한국어로 번역하는 함수
    """
    try:
        if pd.isna(text) or str(text).strip() == '':
            return ''
        
        # 텍스트 전처리
        text = str(text).strip()
        
        # 이미 한국어가 포함되어 있는지 확인 (간단한 휴리스틱)
        korean_chars = len([c for c in text if '\uac00' <= c <= '\ud7af'])
        if korean_chars > len(text) * 0.5:
            return text  # 이미 한국어로 보임
        
        # 텍스트 길이 제한 (Google Translate API 제한)
        if len(text) > 5000:
            text = text[:5000]
        
        # Deep Translator로 번역
        result = translator.translate(text)
        return result
        
    except Exception as e:
        print(f"번역 오류: {str(e)[:100]}")
        return text  # 원본 텍스트 반환

def batch_translate_pros(df, batch_size=50, delay=1.0):
    """
    Pros 데이터를 배치로 번역하는 함수
    """
    # Google Translator 초기화 (deep-translator 사용)
    translator = GoogleTranslator(source='en', target='ko')
    
    # 결과 저장용 리스트
    translated_texts = []
    failed_translations = []
    
    total_rows = len(df)
    print(f"🔄 총 {total_rows:,}개 문장 번역 시작...")
    print(f"📦 배치 크기: {batch_size}, 지연 시간: {delay}초")
    
    # 배치별로 처리
    for i in tqdm(range(0, total_rows, batch_size), desc="번역 진행"):
        batch_end = min(i + batch_size, total_rows)
        batch_data = df.iloc[i:batch_end]
        
        print(f"\n📝 배치 {i//batch_size + 1}: {i+1}~{batch_end} 번역 중...")
        
        batch_translations = []
        for idx, row in batch_data.iterrows():
            sentence = row['pros_sentence']
            
            try:
                translated = translate_text_to_korean(sentence, translator)
                batch_translations.append({
                    'original_index': idx,
                    'original_text': sentence,
                    'korean_text': translated,
                    'translation_status': 'success'
                })
                
            except Exception as e:
                print(f"   ❌ 행 {idx} 번역 실패: {str(e)[:50]}")
                batch_translations.append({
                    'original_index': idx,
                    'original_text': sentence,
                    'korean_text': sentence,  # 원본 유지
                    'translation_status': 'failed'
                })
                failed_translations.append(idx)
        
        translated_texts.extend(batch_translations)
        
        # 배치 간 지연
        if i + batch_size < total_rows:
            print(f"   ⏳ {delay}초 대기 중...")
            time.sleep(delay)
    
    print(f"\n✅ 번역 완료!")
    print(f"   📊 성공: {len(translated_texts) - len(failed_translations):,}개")
    print(f"   ❌ 실패: {len(failed_translations):,}개")
    
    return translated_texts, failed_translations

In [4]:
# 4. 번역 실행 및 진행 상황 모니터링

# 번역 시작 시간 기록
start_time = time.time()
print(f"🚀 번역 시작 시간: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time))}")

# 번역 실행 (배치 크기와 지연 시간 조정 가능)
try:
    translated_results, failed_indices = batch_translate_pros(
        df_pros, 
        batch_size=30,  # 안정성을 위해 작은 배치 크기
        delay=1.5       # Google API 제한을 고려한 지연
    )
    
    # 번역 완료 시간 계산
    end_time = time.time()
    total_time = end_time - start_time
    
    print(f"\n⏰ 총 소요 시간: {total_time/60:.1f}분")
    print(f"⚡ 평균 번역 속도: {len(df_pros)/total_time:.1f}개/초")
    
    # 번역 결과를 DataFrame으로 변환
    df_translated = pd.DataFrame(translated_results)
    
    # 번역 성공률 계산
    success_rate = (len(translated_results) - len(failed_indices)) / len(translated_results) * 100
    print(f"📈 번역 성공률: {success_rate:.1f}%")
    
    # 번역 결과 샘플 확인
    print(f"\n📝 번역 결과 샘플:")
    for i in range(min(3, len(df_translated))):
        row = df_translated.iloc[i]
        print(f"\n   {i+1}. 원문: {row['original_text'][:80]}...")
        print(f"      번역: {row['korean_text'][:80]}...")
        print(f"      상태: {row['translation_status']}")
    
except Exception as e:
    print(f"❌ 번역 중 오류 발생: {e}")
    print("   인터넷 연결과 Google Translate 서비스 상태를 확인해주세요.")

🚀 번역 시작 시간: 2025-08-25 15:51:05
🔄 총 1,129개 문장 번역 시작...
📦 배치 크기: 30, 지연 시간: 1.5초


번역 진행:   0%|          | 0/38 [00:00<?, ?it/s]


📝 배치 1: 1~30 번역 중...
   ⏳ 1.5초 대기 중...
   ⏳ 1.5초 대기 중...


번역 진행:   3%|▎         | 1/38 [00:36<22:36, 36.67s/it]


📝 배치 2: 31~60 번역 중...
   ⏳ 1.5초 대기 중...
   ⏳ 1.5초 대기 중...


...


📝 배치 37: 1081~1110 번역 중...
   ⏳ 1.5초 대기 중...
   ⏳ 1.5초 대기 중...


번역 진행:  97%|█████████▋| 37/38 [22:39<00:30, 30.75s/it]


📝 배치 38: 1111~1129 번역 중...


번역 진행: 100%|██████████| 38/38 [22:56<00:00, 36.22s/it]


✅ 번역 완료!
   📊 성공: 1,129개
   ❌ 실패: 0개

⏰ 총 소요 시간: 22.9분
⚡ 평균 번역 속도: 0.8개/초
📈 번역 성공률: 100.0%

📝 번역 결과 샘플:

   1. 원문: My options seem to be a LG Gram 16, but I do not remember anymore what model was...
      번역: 내 옵션은 LG Gram 16 인 것처럼 보이지만 더 이상 어떤 모델이 좋은 모델인지를 기억하지 못합니다 ... 또는 Yoga Pro 9i Ge...
      상태: success

   2. 원문: Tuning LG Gram Pro 17 for productivity? Lack Snappy feeling..  Any Tips?...
      번역: 생산성을 위해 LG Gram Pro 17을 조정합니까? 칙칙한 느낌이 부족합니다 .. 어떤 팁이 있습니까?...
      상태: success

   3. 원문: I have LG Gram Pro 17 32GB Ultra 7. Even setting FAN to HIGH, Turned off Energy ...
      번역: 나는 LG Gram Pro 17 32GB Ultra 7을 가지고 있습니다. 팬을 높음으로 설정하고 에너지 절약 기능을 끄했습니다....
      상태: success


In [5]:
# 5. 번역 결과 검증 및 품질 확인

def analyze_translation_quality(df_translated):
    """번역 품질을 분석하는 함수"""
    
    print("🔍 번역 품질 분석 중...")
    
    # 기본 통계
    total_count = len(df_translated)
    success_count = len(df_translated[df_translated['translation_status'] == 'success'])
    failed_count = total_count - success_count
    
    print(f"\n📊 번역 통계:")
    print(f"   📝 전체 문장: {total_count:,}개")
    print(f"   ✅ 성공 번역: {success_count:,}개 ({success_count/total_count*100:.1f}%)")
    print(f"   ❌ 실패 번역: {failed_count:,}개 ({failed_count/total_count*100:.1f}%)")
    
    # 성공한 번역만 분석
    success_df = df_translated[df_translated['translation_status'] == 'success'].copy()
    
    if len(success_df) > 0:
        # 문장 길이 분석
        success_df['original_length'] = success_df['original_text'].str.len()
        success_df['korean_length'] = success_df['korean_text'].str.len()
        success_df['length_ratio'] = success_df['korean_length'] / success_df['original_length']
        
        print(f"\n📏 문장 길이 분석:")
        print(f"   📝 원문 평균 길이: {success_df['original_length'].mean():.1f}자")
        print(f"   🇰🇷 번역 평균 길이: {success_df['korean_length'].mean():.1f}자")
        print(f"   📊 길이 비율: {success_df['length_ratio'].mean():.2f}")
        
        # 한국어 문자 비율 확인
        korean_char_ratios = []
        for text in success_df['korean_text']:
            korean_chars = len([c for c in str(text) if '\uac00' <= c <= '\ud7af'])
            total_chars = len(str(text))
            if total_chars > 0:
                ratio = korean_chars / total_chars
                korean_char_ratios.append(ratio)
        
        if korean_char_ratios:
            avg_korean_ratio = sum(korean_char_ratios) / len(korean_char_ratios)
            print(f"   🇰🇷 한국어 문자 비율: {avg_korean_ratio*100:.1f}%")
        
        # 번역 품질 샘플 확인
        print(f"\n🔍 번역 품질 샘플 (무작위 5개):")
        sample_indices = np.random.choice(len(success_df), min(5, len(success_df)), replace=False)
        
        for i, idx in enumerate(sample_indices):
            row = success_df.iloc[idx]
            print(f"\n   {i+1}. 원문: {row['original_text']}")
            print(f"      번역: {row['korean_text']}")
            print(f"      길이: {row['original_length']}자 → {row['korean_length']}자")
    
    return success_df

# 번역 품질 분석 실행
if 'df_translated' in locals():
    quality_df = analyze_translation_quality(df_translated)
else:
    print("❌ 번역 결과가 없습니다. 이전 셀을 먼저 실행해주세요.")

🔍 번역 품질 분석 중...

📊 번역 통계:
   📝 전체 문장: 1,129개
   ✅ 성공 번역: 1,129개 (100.0%)
   ❌ 실패 번역: 0개 (0.0%)

📏 문장 길이 분석:
   📝 원문 평균 길이: 238.9자
   🇰🇷 번역 평균 길이: 156.8자
   📊 길이 비율: 0.73
   🇰🇷 한국어 문자 비율: 32.3%

🔍 번역 품질 샘플 (무작위 5개):

   1. 원문: I will be joining uni the next month, and I'm planning to buy a decent laptop for programming, research and light games, I was initially planning on getting the macbook pro with the m4 pro chip but on further research I realised that apple silicon doesn't allow bootcamp(since they use ARM arch now), and plus I feel that I will face compatibility issues in the future since I'm a cs major, so I'm looking for a laptop that beats the macbook pro or is at least comparable in certain aspects such as battery life, screen and performance. I ended up finding the lenovo yoga pro 9i, with rtx 4050 32 gb ram, but its battery life is not that great(approx 7 hrs), and the asus zephyrus g16(2024) with rtx 4070 and 32 gb ram, it provides decent battery life and a good screen. is 

In [6]:
# 6. 최종 결과 저장 및 요약 보고서 생성

def save_translation_results(df_translated, output_path):
    """번역 결과를 CSV 파일로 저장"""
    
    try:
        # 출력 디렉토리 생성
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # 번역 결과 정리
        final_df = pd.DataFrame({
            'pros_sentence_english': df_translated['original_text'],
            'pros_sentence_korean': df_translated['korean_text'],
            'translation_status': df_translated['translation_status']
        })
        
        # CSV로 저장
        final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 번역 결과 저장 완료: {output_path}")
        
        return final_df
        
    except Exception as e:
        print(f"❌ 파일 저장 중 오류: {e}")
        return None

def generate_translation_report(df_translated, final_df, output_dir):
    """번역 작업 요약 보고서 생성"""
    
    report_content = f"""# LG Gram Pros 분석 데이터 한국어 번역 보고서

## 📊 번역 작업 개요
- **번역 일시**: {time.strftime('%Y-%m-%d %H:%M:%S')}
- **번역 도구**: Google Translate API (deep-translator)
- **번역 방향**: 영어 → 한국어

## 📈 번역 통계
- **전체 문장 수**: {len(df_translated):,}개
- **성공 번역**: {len(df_translated[df_translated['translation_status'] == 'success']):,}개
- **실패 번역**: {len(df_translated[df_translated['translation_status'] == 'failed']):,}개
- **성공률**: {len(df_translated[df_translated['translation_status'] == 'success'])/len(df_translated)*100:.1f}%

## 📁 생성된 파일
- **원본 파일**: pros_analysis.csv
- **번역 파일**: Pros_LG_Korean.csv
- **컬럼 구성**: 
  - pros_sentence_english (원문)
  - pros_sentence_korean (번역)
  - translation_status (번역 상태)

## 🔍 번역 품질 분석
"""
    
    # 성공한 번역 데이터로 품질 분석
    success_df = df_translated[df_translated['translation_status'] == 'success']
    if len(success_df) > 0:
        success_df_copy = success_df.copy()
        success_df_copy['original_length'] = success_df_copy['original_text'].str.len()
        success_df_copy['korean_length'] = success_df_copy['korean_text'].str.len()
        
        report_content += f"""
- **평균 원문 길이**: {success_df_copy['original_length'].mean():.1f}자
- **평균 번역 길이**: {success_df_copy['korean_length'].mean():.1f}자
- **길이 비율**: {(success_df_copy['korean_length']/success_df_copy['original_length']).mean():.2f}

## 📝 번역 샘플
"""
        
        # 번역 샘플 추가
        for i, (idx, row) in enumerate(success_df.head(3).iterrows()):
            report_content += f"""
### 샘플 {i+1}
- **원문**: {row['original_text']}
- **번역**: {row['korean_text']}
"""
    
    report_content += f"""

## ✅ 작업 완료 사항
1. ✅ 영어 Pros 데이터 로드 ({len(df_translated):,}개 문장)
2. ✅ Google Translate API를 통한 한국어 번역
3. ✅ 번역 품질 검증 및 분석
4. ✅ 한국어 번역 결과 CSV 파일 생성
5. ✅ 번역 작업 보고서 생성

## 📋 다음 단계 제안
1. 번역된 한국어 데이터를 활용한 감정 분석
2. 한국어 키워드 추출 및 빈도 분석
3. 영어-한국어 번역 결과 비교 분석
4. LG Gram 제품 장점 강화를 위한 한국어 텍스트 마이닝

---
*보고서 생성 시간: {time.strftime('%Y-%m-%d %H:%M:%S')}*
"""
    
    # 보고서 파일 저장
    report_path = output_dir / 'LG_Gram_Pros_Translation_Report.md'
    try:
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report_content)
        print(f"📋 번역 보고서 생성 완료: {report_path}")
    except Exception as e:
        print(f"❌ 보고서 생성 중 오류: {e}")

# 번역 결과 저장 및 보고서 생성
if 'df_translated' in locals() and 'korean_pros_file' in locals():
    print("💾 번역 결과 저장 중...")
    
    # CSV 파일 저장
    final_result_df = save_translation_results(df_translated, korean_pros_file)
    
    if final_result_df is not None:
        print(f"\n📋 최종 결과 요약:")
        print(f"   📄 저장된 파일: {korean_pros_file}")
        print(f"   📊 총 데이터: {len(final_result_df):,}행")
        print(f"   📂 파일 크기: {korean_pros_file.stat().st_size / 1024:.1f} KB")
        
        # 번역 보고서 생성
        generate_translation_report(df_translated, final_result_df, korean_pros_file.parent)
        
        print(f"\n🎉 모든 번역 작업이 완료되었습니다!")
        print(f"   ✅ 번역 파일: Pros_LG_Korean.csv")
        print(f"   📋 보고서: LG_Gram_Pros_Translation_Report.md")
        
    else:
        print("❌ 번역 결과 저장에 실패했습니다.")
        
else:
    print("❌ 번역 데이터나 출력 경로가 설정되지 않았습니다.")
    print("   이전 셀들을 순서대로 실행해주세요.")

💾 번역 결과 저장 중...
✅ 번역 결과 저장 완료: c:\Users\lgdx\LG_DX_School\03.CX_Group4\02.LG_Gram\data\lg_gram_reddit\analysis\Pros_LG_Korean.csv

📋 최종 결과 요약:
   📄 저장된 파일: c:\Users\lgdx\LG_DX_School\03.CX_Group4\02.LG_Gram\data\lg_gram_reddit\analysis\Pros_LG_Korean.csv
   📊 총 데이터: 1,129행
   📂 파일 크기: 570.4 KB
📋 번역 보고서 생성 완료: c:\Users\lgdx\LG_DX_School\03.CX_Group4\02.LG_Gram\data\lg_gram_reddit\analysis\LG_Gram_Pros_Translation_Report.md

🎉 모든 번역 작업이 완료되었습니다!
   ✅ 번역 파일: Pros_LG_Korean.csv
   📋 보고서: LG_Gram_Pros_Translation_Report.md
